# Hallway Illuminance Accessible Datasets Train/Eval Notebook

This notebook is a new notebook-first workflow for hallway illuminance estimation using only the more accessible dataset path:
- NYU Depth V2 via `kagglehub`
- MID Intrinsics
- Fast Spatially-Varying Indoor Lighting Estimation
- optional custom hallway dataset

`MIT Intrinsic Images` is intentionally not used in this notebook.


## 1. Project Overview

This notebook keeps the practical train/validate/test flow in one place while using package modules for the real implementation.

Dataset policy in this notebook:
- `NYU Depth V2` is pulled from Kaggle by default.
- `MID Intrinsics` is the main accessible intrinsic prior.
- `Fast Spatially-Varying Indoor Lighting Estimation` is used for local lighting priors.
- `custom hallway dataset` remains the most important source for actual lux maps, point targets, and power-derived carbon outputs.

`MIT Intrinsic Images` is excluded here because it is awkward to obtain reliably.

If you want another intrinsic-style dataset later, `CGIntrinsics` is the next reasonable addition, but it is not wired into the package registry in this notebook.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT_OVERRIDE = ''


def detect_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE:
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if not (candidate / 'configs').exists() or not (candidate / 'hallway_lighting').exists():
            raise FileNotFoundError(f'PROJECT_ROOT_OVERRIDE does not point to the repo root: {candidate}')
        return candidate

    for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
        if (candidate / 'configs').exists() and (candidate / 'hallway_lighting').exists():
            return candidate

    raise FileNotFoundError(
        'Could not detect the repo root automatically. Set PROJECT_ROOT_OVERRIDE to the hallway_lighting repo root.'
    )


PROJECT_ROOT = detect_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'Package dir exists: {(PROJECT_ROOT / "hallway_lighting").exists()}')
print(f'Configs dir exists: {(PROJECT_ROOT / "configs").exists()}')


## 2. Dependency Installation

Run the install step once per Colab session. `requirements.txt` includes `kagglehub` for NYU download and `onnxruntime` for ONNX-side inference checks.


In [ ]:
requirements_file = PROJECT_ROOT / 'requirements.txt'
print(f'Install dependencies from: {requirements_file}')
# In Colab, uncomment the next line:
# %pip install -q -r ../requirements.txt


## 3. Mount Google Drive

Drive is still useful for custom hallway data, cached exports, and checkpoints. NYU Depth V2 can come from Kaggle directly and does not need Drive.


In [ ]:
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    print('google.colab is not available in this environment.')

print(f'Running in Colab: {IN_COLAB}')


## 4. Load Configs And Create Run Directories

This notebook writes manifests, checkpoints, inference outputs, and exports under a dedicated run directory.


In [ ]:
import pandas as pd

from hallway_lighting.utils.io import ensure_dir, load_yaml, save_config_snapshot

base_config = load_yaml(PROJECT_ROOT / 'configs' / 'base.yaml')
datasets_config = load_yaml(PROJECT_ROOT / 'configs' / 'datasets.yaml')
train_config = load_yaml(PROJECT_ROOT / 'configs' / 'train.yaml')
carbon_config = load_yaml(PROJECT_ROOT / 'configs' / 'carbon.yaml')
infer_config = load_yaml(PROJECT_ROOT / 'configs' / 'infer.yaml')

RUN_ROOT = ensure_dir(PROJECT_ROOT / 'runs' / 'accessible_datasets_notebook')
MANIFEST_DIR = ensure_dir(RUN_ROOT / 'manifests')
PREPARED_ROOT = ensure_dir(RUN_ROOT / 'prepared_datasets')
CHECKPOINT_DIR = ensure_dir(RUN_ROOT / 'checkpoints')
VIS_DIR = ensure_dir(RUN_ROOT / 'visualizations')
CONFIG_SNAPSHOT_DIR = ensure_dir(RUN_ROOT / 'config_snapshot')
HISTORY_PATH = RUN_ROOT / 'training_history.json'
BEST_MODEL_PATH = CHECKPOINT_DIR / 'best_model.pt'
LAST_MODEL_PATH = CHECKPOINT_DIR / 'last_model.pt'

save_config_snapshot(
    {
        'base': base_config,
        'datasets': datasets_config,
        'train': train_config,
        'carbon': carbon_config,
        'infer': infer_config,
    },
    CONFIG_SNAPSHOT_DIR,
)

RUN_ROOT


## 5. Dataset Inputs

This notebook only uses the accessible integrated datasets:
- `nyu_depth_v2`
- `mid_intrinsics`
- `fast_sv_indoor_lighting`
- `custom_hallway`

`nyu_depth_v2` is downloaded from Kaggle by default. Fill in the remaining paths before training. If you have custom hallway labels, set that path first because it matters most for real hallway lux outputs.


In [ ]:
from pathlib import Path

USE_KAGGLE_NYU = True
NYU_KAGGLE_DATASET = 'soumikrakshit/nyu-depth-v2'

DATASET_INPUTS = {
    'nyu_depth_v2': datasets_config['dataset_roots'].get('nyu_depth_v2', ''),
    'mid_intrinsics': datasets_config['dataset_roots'].get('mid_intrinsics', ''),
    'fast_sv_indoor_lighting': datasets_config['dataset_roots'].get('fast_sv_indoor_lighting', ''),
    'custom_hallway': datasets_config['dataset_roots'].get('custom_hallway', ''),
}

if USE_KAGGLE_NYU:
    import kagglehub

    nyu_kaggle_path = Path(kagglehub.dataset_download(NYU_KAGGLE_DATASET))
    DATASET_INPUTS['nyu_depth_v2'] = str(nyu_kaggle_path)
    print(f'NYU Depth V2 downloaded from Kaggle to: {nyu_kaggle_path}')
elif not DATASET_INPUTS['nyu_depth_v2']:
    print("Set DATASET_INPUTS['nyu_depth_v2'] manually if you are not using Kaggle.")

# Example manual overrides:
# DATASET_INPUTS['mid_intrinsics'] = '/content/drive/MyDrive/datasets/MIDIntrinsics'
# DATASET_INPUTS['fast_sv_indoor_lighting'] = '/content/drive/MyDrive/datasets/fast_sv_indoor_lighting'
# DATASET_INPUTS['custom_hallway'] = '/content/drive/MyDrive/datasets/custom_hallway/custom_hallway_manifest.csv'

pd.DataFrame(
    {
        'dataset_name': list(DATASET_INPUTS.keys()),
        'input_path': list(DATASET_INPUTS.values()),
    }
)


## 6. Prepare Inputs And Build Manifests

Every dataset input can be an extracted folder, archive, `.mat`, or custom hallway CSV. The Kaggle NYU directory is treated like any other dataset root.


In [ ]:
from hallway_lighting.data.archive_utils import prepare_dataset_input
from hallway_lighting.data.dataset_registry import build_all_dataset_manifests

PREPARED_DATASET_INPUTS = {}
for dataset_name, input_path in DATASET_INPUTS.items():
    if not str(input_path).strip():
        continue
    PREPARED_DATASET_INPUTS[dataset_name] = prepare_dataset_input(
        input_path=input_path,
        dataset_name=dataset_name,
        working_dir=PREPARED_ROOT,
        overwrite=False,
    )

prepared_rows = [
    {
        'dataset_name': item.dataset_name,
        'input_type': item.input_type,
        'source_path': str(item.source_path),
        'prepared_root': str(item.prepared_root),
        'primary_file': '' if item.primary_file is None else str(item.primary_file),
    }
    for item in PREPARED_DATASET_INPUTS.values()
]

display(pd.DataFrame(prepared_rows)) if prepared_rows else print('No dataset inputs configured yet.')

MANIFEST_RESULTS = build_all_dataset_manifests(
    dataset_inputs=DATASET_INPUTS,
    working_dir=RUN_ROOT,
    output_dir=MANIFEST_DIR,
    overwrite=False,
)
MANIFESTS = {name: result.manifest for name, result in MANIFEST_RESULTS.items()}
MANIFEST_PATHS = {name: result.manifest_path for name, result in MANIFEST_RESULTS.items()}

pd.DataFrame(
    {
        'dataset_name': list(MANIFEST_PATHS.keys()),
        'manifest_path': [str(path) for path in MANIFEST_PATHS.values()],
    }
)


## 7. Manifest Diagnostics

Review split counts and label coverage before training. This makes it obvious when a dataset only contributes priors versus direct hallway supervision.


In [ ]:
manifest_summary_rows = []
for dataset_name, manifest in MANIFESTS.items():
    manifest_summary_rows.append(
        {
            'dataset_name': dataset_name,
            'rows': int(len(manifest)),
            'train_rows': int((manifest['split'].fillna('').str.lower() == 'train').sum()),
            'val_rows': int((manifest['split'].fillna('').str.lower() == 'val').sum()),
            'test_rows': int((manifest['split'].fillna('').str.lower() == 'test').sum()),
            'has_lux_map': int(manifest['lux_map_path'].fillna('').astype(str).str.len().gt(0).sum()) if 'lux_map_path' in manifest else 0,
            'has_floor_mask': int(manifest['floor_mask_path'].fillna('').astype(str).str.len().gt(0).sum()) if 'floor_mask_path' in manifest else 0,
            'has_albedo': int(manifest['albedo_path'].fillna('').astype(str).str.len().gt(0).sum()) if 'albedo_path' in manifest else 0,
            'has_gloss': int(manifest['gloss_path'].fillna('').astype(str).str.len().gt(0).sum()) if 'gloss_path' in manifest else 0,
            'has_points': int(manifest['point_targets_json'].fillna('').astype(str).str.len().gt(0).sum()) if 'point_targets_json' in manifest else 0,
        }
    )

display(pd.DataFrame(manifest_summary_rows)) if manifest_summary_rows else print('No manifests were built.')

for dataset_name, manifest in MANIFESTS.items():
    print(f'Preview: {dataset_name}')
    display(manifest.head(3))


## 8. Sample Visualization

Show one accessible sample before training. If custom hallway data is available, that is usually the most informative sample to inspect first.


In [ ]:
import numpy as np
from PIL import Image

from hallway_lighting.data.point_sampling import default_hallway_points
from hallway_lighting.utils.visualization import overlay_points, show_image

sample_row = None
for dataset_name in ('custom_hallway', 'mid_intrinsics', 'fast_sv_indoor_lighting', 'nyu_depth_v2'):
    manifest = MANIFESTS.get(dataset_name)
    if manifest is None or manifest.empty:
        continue
    valid_rows = manifest.loc[manifest['image_path'].fillna('').astype(str).str.len().gt(0)]
    if valid_rows.empty:
        continue
    sample_row = valid_rows.iloc[0]
    break

if sample_row is None:
    print('No sample row available for visualization yet.')
else:
    image_path = Path(sample_row['image_path'])
    sample_image = np.asarray(Image.open(image_path).convert('RGB'))
    show_image(sample_image, title=f"Sample: {sample_row['dataset_name']}")
    if sample_row['dataset_name'] == 'custom_hallway':
        overlay_points(sample_image, default_hallway_points(), title='Canonical Hallway Points')


## 9. Build Dataloaders

This notebook uses the package training helpers to build multi-dataset train, validation, and test loaders from normalized manifests.


In [ ]:
from hallway_lighting.training import build_dataloaders

IMAGE_SIZE = (
    int(train_config['training']['image_size']['height']),
    int(train_config['training']['image_size']['width']),
)

DATALOADERS = build_dataloaders(
    manifests=MANIFESTS,
    batch_size=int(train_config['training']['batch_size']),
    num_workers=int(train_config['training']['num_workers']),
    image_size=IMAGE_SIZE,
    seed=int(base_config['project']['seed']),
)

train_loader = DATALOADERS['train']
val_loader = DATALOADERS['val']
test_loader = DATALOADERS['test']

pd.DataFrame(
    {
        'split': ['train', 'val', 'test'],
        'available': [loader is not None for loader in (train_loader, val_loader, test_loader)],
        'num_batches': [0 if loader is None else len(loader) for loader in (train_loader, val_loader, test_loader)],
    }
)


## 10. Inspect One Batch

Check one collated batch before training so missing-label masking is easier to reason about.


In [ ]:
if train_loader is None:
    print('No train loader available yet.')
else:
    first_batch = next(iter(train_loader))
    batch_summary = {
        'image_shape': tuple(first_batch['image'].shape),
        'datasets_in_batch': list(first_batch['dataset_name']),
        'has_lux_map_targets': first_batch['lux_map'] is not None,
        'has_floor_targets': first_batch['floor_mask'] is not None,
        'has_albedo_targets': first_batch['albedo'] is not None,
        'has_gloss_targets': first_batch['gloss'] is not None,
        'has_point_targets': first_batch['point_lux'] is not None,
    }
    batch_summary


## 11. Initialize Model, Optimizer, Scheduler, And AMP

This section creates the real multitask model and all notebook-side training state.


In [ ]:
import torch

from hallway_lighting.models.hallway_multitask_unet import HallwayMultitaskUNet
from hallway_lighting.utils.io import load_checkpoint, save_checkpoint, save_training_history
from hallway_lighting.utils.seed import set_seed

set_seed(int(base_config['project']['seed']))

device_name = str(train_config['training'].get('device', 'cuda'))
device = torch.device('cuda' if device_name == 'cuda' and torch.cuda.is_available() else 'cpu')
amp_enabled = bool(train_config['training'].get('amp', True)) and device.type == 'cuda'

model = HallwayMultitaskUNet(base_config['model']).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(train_config['training']['learning_rate']),
    weight_decay=float(train_config['training']['weight_decay']),
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(int(train_config['training']['epochs']), 1),
)
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

HISTORY = {'train': [], 'val': [], 'test': []}
START_EPOCH = 0
BEST_VAL_SCORE = float('inf')
TRAIN_RESULT = None
VAL_RESULT = None
TEST_RESULT = None


def count_parameters(module: torch.nn.Module) -> dict:
    total_params = sum(parameter.numel() for parameter in module.parameters())
    trainable_params = sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)
    return {'total_parameters': total_params, 'trainable_parameters': trainable_params}


def load_named_checkpoint(checkpoint_name: str = 'best') -> None:
    global HISTORY, START_EPOCH, BEST_VAL_SCORE
    checkpoint_path = BEST_MODEL_PATH if checkpoint_name == 'best' else LAST_MODEL_PATH
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

    checkpoint = load_checkpoint(
        checkpoint_path,
        model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        map_location=device,
    )
    HISTORY = checkpoint.get('history', HISTORY)
    START_EPOCH = int(checkpoint.get('epoch', -1)) + 1
    BEST_VAL_SCORE = float(checkpoint.get('extra_state', {}).get('best_val_score', BEST_VAL_SCORE))
    print(f'Loaded {checkpoint_name} checkpoint from {checkpoint_path}')


parameter_summary = count_parameters(model)
parameter_summary


## 12. Training Controls

This section includes a fast 'tonight run' profile. By default it is set up to train first, then let you run validation and testing after a checkpoint exists.


In [ ]:
from hallway_lighting.training import run_epoch

TONIGHT_RUN_PROFILE = True

if TONIGHT_RUN_PROFILE:
    RUN_TRAINING = True
    RUN_VALIDATION = False
    RUN_TEST = False
    EPOCHS_OVERRIDE = 10
else:
    RUN_TRAINING = False
    RUN_VALIDATION = False
    RUN_TEST = False
    EPOCHS_OVERRIDE = None

TRAINING_EPOCHS = int(EPOCHS_OVERRIDE or train_config['training']['epochs'])
GRAD_CLIP = float(train_config['optimization']['gradient_clip_norm'])
MAX_VIS_EXAMPLES = int(train_config['training']['max_visualization_examples'])
VALIDATE_EVERY = int(train_config['training']['validate_every'])
SAVE_EVERY = int(train_config['training']['save_every'])
LOSS_WEIGHTS = train_config['loss_weights']

{
    'tonight_run_profile': TONIGHT_RUN_PROFILE,
    'run_training': RUN_TRAINING,
    'run_validation': RUN_VALIDATION,
    'run_test': RUN_TEST,
    'device': str(device),
    'amp_enabled': amp_enabled,
    'training_epochs': TRAINING_EPOCHS,
    'validate_every': VALIDATE_EVERY,
    'save_every': SAVE_EVERY,
}



## 13. Training

This loop uses dataset-specific supervision routing from the package. Missing targets are skipped automatically through batch availability masks.


In [ ]:
if RUN_TRAINING:
    if train_loader is None:
        raise RuntimeError('train_loader is not available. Build manifests and dataloaders first.')

    for epoch in range(START_EPOCH, TRAINING_EPOCHS):
        TRAIN_RESULT = run_epoch(
            model=model,
            dataloader=train_loader,
            device=device,
            loss_weights=LOSS_WEIGHTS,
            carbon_config=carbon_config,
            optimizer=optimizer,
            scaler=scaler,
            amp_enabled=amp_enabled,
            max_visualization_examples=MAX_VIS_EXAMPLES,
            gradient_clip_norm=GRAD_CLIP,
        )
        train_summary = {'epoch': epoch + 1, **TRAIN_RESULT.summary}
        HISTORY['train'].append(train_summary)

        if scheduler is not None:
            scheduler.step()

        monitored_score = float(train_summary.get('total_loss', 0.0))
        if val_loader is not None and (epoch + 1) % VALIDATE_EVERY == 0:
            VAL_RESULT = run_epoch(
                model=model,
                dataloader=val_loader,
                device=device,
                loss_weights=LOSS_WEIGHTS,
                carbon_config=carbon_config,
                optimizer=None,
                scaler=None,
                amp_enabled=amp_enabled,
                max_visualization_examples=MAX_VIS_EXAMPLES,
                gradient_clip_norm=GRAD_CLIP,
            )
            val_summary = {'epoch': epoch + 1, **VAL_RESULT.summary}
            HISTORY['val'].append(val_summary)
            monitored_score = float(val_summary.get('total_loss', monitored_score))

        if (epoch + 1) % SAVE_EVERY == 0:
            save_checkpoint(
                model=model,
                optimizer=optimizer,
                epoch=epoch,
                path=LAST_MODEL_PATH,
                scheduler=scheduler,
                scaler=scaler,
                history=HISTORY,
                extra_state={'best_val_score': BEST_VAL_SCORE},
            )

        if monitored_score <= BEST_VAL_SCORE:
            BEST_VAL_SCORE = monitored_score
            save_checkpoint(
                model=model,
                optimizer=optimizer,
                epoch=epoch,
                path=BEST_MODEL_PATH,
                scheduler=scheduler,
                scaler=scaler,
                history=HISTORY,
                extra_state={'best_val_score': BEST_VAL_SCORE},
            )

        save_training_history(HISTORY, HISTORY_PATH)
        print(
            f"Epoch {epoch + 1}/{TRAINING_EPOCHS} | "
            f"train_total_loss={train_summary.get('total_loss', 0.0):.4f} | "
            f"best_val_score={BEST_VAL_SCORE:.4f}"
        )

    START_EPOCH = TRAINING_EPOCHS
else:
    print('Set RUN_TRAINING = True to execute training.')


## 14. Validation

Run validation on demand outside the training loop when you want a clean pass from the current checkpoint or in-memory model.


In [ ]:
if RUN_VALIDATION:
    if val_loader is None:
        raise RuntimeError('val_loader is not available.')

    VAL_RESULT = run_epoch(
        model=model,
        dataloader=val_loader,
        device=device,
        loss_weights=LOSS_WEIGHTS,
        carbon_config=carbon_config,
        optimizer=None,
        scaler=None,
        amp_enabled=amp_enabled,
        max_visualization_examples=MAX_VIS_EXAMPLES,
        gradient_clip_norm=GRAD_CLIP,
    )
    validation_summary = pd.DataFrame([VAL_RESULT.summary])
    display(validation_summary)
else:
    print('Set RUN_VALIDATION = True to execute validation.')


## 15. Testing / Evaluation

Use the held-out split when available. This notebook keeps evaluation in the same environment as training so it is easy to inspect examples immediately.


In [ ]:
if RUN_TEST:
    if test_loader is None:
        raise RuntimeError('test_loader is not available.')

    TEST_RESULT = run_epoch(
        model=model,
        dataloader=test_loader,
        device=device,
        loss_weights=LOSS_WEIGHTS,
        carbon_config=carbon_config,
        optimizer=None,
        scaler=None,
        amp_enabled=amp_enabled,
        max_visualization_examples=MAX_VIS_EXAMPLES,
        gradient_clip_norm=GRAD_CLIP,
    )
    test_summary = pd.DataFrame([TEST_RESULT.summary])
    display(test_summary)
else:
    print('Set RUN_TEST = True to execute testing.')


## 16. Visualize Predictions

This section saves and displays one prediction overview from the latest available validation or test result.


In [ ]:
from IPython.display import Image as NotebookImage

from hallway_lighting.utils.visualization import save_prediction_figure

RESULT_FOR_VIS = TEST_RESULT if TEST_RESULT is not None else VAL_RESULT
if RESULT_FOR_VIS is None or not RESULT_FOR_VIS.visual_examples:
    print('Run validation or testing first to populate visual examples.')
else:
    example = RESULT_FOR_VIS.visual_examples[0]
    figure_path = VIS_DIR / f"{example['sample_id']}_prediction.png"
    save_prediction_figure(
        path=figure_path,
        image=example['image'],
        lux_map=example['lux_map_pred'],
        floor_mask_pred=example['floor_mask_pred'],
        albedo_pred=example['albedo_pred'],
        gloss_pred=example['gloss_pred'],
        points=example['point_targets'],
        point_values=example['point_lux_pred'],
        title=f"Prediction Overview: {example['sample_id']}",
    )
    display(NotebookImage(filename=str(figure_path)))
    print(f'Saved figure: {figure_path}')


## 17. Point-wise Illuminance Reporting

Point-wise hallway outputs are sampled from the predicted lux map at canonical floor points under fixtures and between fixtures.


In [ ]:
POINT_REPORT_SOURCE = TEST_RESULT if TEST_RESULT is not None else VAL_RESULT
if POINT_REPORT_SOURCE is None or not POINT_REPORT_SOURCE.point_reports:
    print('Run validation or testing on data with point targets to populate point reports.')
else:
    point_report_df = pd.DataFrame(POINT_REPORT_SOURCE.point_reports)
    display(point_report_df)


## 18. Carbon Reporting

Carbon outputs are derived from lighting electricity use, not directly sensed. This section builds a report from the latest available predicted lux map.


In [ ]:
from hallway_lighting.utils.carbon import summarize_carbon_from_lux
from hallway_lighting.utils.metrics import summarize_lux_map

avg_lux_for_carbon = 150.0
RESULT_FOR_CARBON = TEST_RESULT if TEST_RESULT is not None else VAL_RESULT
if RESULT_FOR_CARBON is not None and RESULT_FOR_CARBON.visual_examples:
    avg_lux_for_carbon = summarize_lux_map(RESULT_FOR_CARBON.visual_examples[0]['lux_map_pred'])['avg_lux']

carbon_report = summarize_carbon_from_lux(
    avg_lux=avg_lux_for_carbon,
    floor_area_m2=float(infer_config['inference']['floor_area_m2']),
    watts_per_lux_m2=float(carbon_config['carbon']['watts_per_lux_m2']),
    interval_hours=float(carbon_config['carbon']['default_interval_hours']),
    carbon_factor_kg_per_kwh=float(carbon_config['carbon']['default_grid_carbon_factor_kg_per_kwh']),
)
carbon_report


## 19. Checkpoints And Training History

Inspect saved checkpoints and the training-history JSON produced by the notebook.


In [ ]:
import json
import matplotlib.pyplot as plt

checkpoint_rows = [
    {'checkpoint_file': path.name, 'path': str(path)}
    for path in sorted(CHECKPOINT_DIR.glob('*.pt'))
]
display(pd.DataFrame(checkpoint_rows)) if checkpoint_rows else print('No checkpoints saved yet.')

if HISTORY_PATH.exists():
    history_payload = json.loads(HISTORY_PATH.read_text())
    train_history_df = pd.DataFrame(history_payload.get('train', []))
    val_history_df = pd.DataFrame(history_payload.get('val', []))
    test_history_df = pd.DataFrame(history_payload.get('test', []))

    if not train_history_df.empty:
        display(train_history_df.tail())
        plt.figure(figsize=(8, 4))
        plt.plot(train_history_df['epoch'], train_history_df['total_loss'], label='train_total_loss')
        if not val_history_df.empty and 'total_loss' in val_history_df:
            plt.plot(val_history_df['epoch'], val_history_df['total_loss'], label='val_total_loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training History')
        plt.legend()
        plt.tight_layout()
        plt.show()
    if not test_history_df.empty:
        display(test_history_df.tail())
else:
    print('No saved training history yet.')


## 20. ONNX Export

Export the multitask model with the shared package helper. This ONNX file is the main deployment-oriented artifact for later Raspberry Pi work.


In [ ]:
from hallway_lighting.infer import export_model_to_onnx, load_model_from_checkpoint


def resolve_project_path(path_value: str) -> Path:
    path = Path(path_value)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


export_settings = infer_config['inference']
export_checkpoint_value = str(export_settings.get('checkpoint_path', '')).strip()
if export_checkpoint_value:
    export_checkpoint_path = resolve_project_path(export_checkpoint_value)
elif BEST_MODEL_PATH.exists():
    export_checkpoint_path = BEST_MODEL_PATH
elif LAST_MODEL_PATH.exists():
    export_checkpoint_path = LAST_MODEL_PATH
else:
    export_checkpoint_path = None

onnx_path = resolve_project_path(export_settings['export_onnx_path'])

if export_checkpoint_path is None or not export_checkpoint_path.exists():
    print('No checkpoint available yet. Train the model first or set inference.checkpoint_path in configs/infer.yaml.')
else:
    export_model = load_model_from_checkpoint(
        checkpoint_path=export_checkpoint_path,
        model_config=base_config['model'],
        device='cpu',
    )
    exported_path = export_model_to_onnx(
        model=export_model,
        output_path=onnx_path,
        image_size=IMAGE_SIZE,
        device='cpu',
        opset_version=int(export_settings.get('opset_version', 17)),
    )
    print(f'Exported ONNX model to: {exported_path}')


## 21. Single-Image Inference

Run single-image inference from either the current PyTorch model or a saved checkpoint. The helper saves JSON, heatmap, overlay, and point-annotation artifacts.


In [ ]:
import json
from IPython.display import Image as NotebookImage

from hallway_lighting.data.point_sampling import default_hallway_points
from hallway_lighting.infer import run_single_image_inference

inference_settings = infer_config['inference']
single_image_value = str(inference_settings.get('single_image_path', '')).strip()
inference_output_dir = resolve_project_path(inference_settings['output_dir'])
fixture_count = int(base_config['model']['fixture_count'])

if not single_image_value:
    print('Set inference.single_image_path in configs/infer.yaml or override it in this cell before running inference.')
else:
    single_image_path = resolve_project_path(single_image_value)
    if not single_image_path.exists():
        raise FileNotFoundError(f'Single-image path does not exist: {single_image_path}')

    checkpoint_value = str(inference_settings.get('checkpoint_path', '')).strip()
    if checkpoint_value:
        checkpoint_path = resolve_project_path(checkpoint_value)
    elif BEST_MODEL_PATH.exists():
        checkpoint_path = BEST_MODEL_PATH
    elif LAST_MODEL_PATH.exists():
        checkpoint_path = LAST_MODEL_PATH
    else:
        checkpoint_path = None

    common_kwargs = dict(
        image_path=single_image_path,
        image_size=IMAGE_SIZE,
        point_targets=default_hallway_points(fixture_count=fixture_count),
        fixture_count=fixture_count,
        output_dir=inference_output_dir,
        save_outputs=bool(inference_settings.get('save_outputs', True)),
        save_point_visualization=bool(inference_settings.get('save_point_visualization', True)),
        floor_area_m2=float(inference_settings['floor_area_m2']),
        watts_per_lux_m2=float(carbon_config['carbon']['watts_per_lux_m2']),
        carbon_factor_kg_per_kwh=float(carbon_config['carbon']['default_grid_carbon_factor_kg_per_kwh']),
        interval_hours=float(inference_settings['interval_hours']),
    )

    if checkpoint_path is not None and checkpoint_path.exists():
        inference_output = run_single_image_inference(
            checkpoint_path=checkpoint_path,
            model_config=base_config['model'],
            device=str(device),
            **common_kwargs,
        )
    else:
        print('No saved checkpoint found. Running inference with the in-memory model parameters.')
        inference_output = run_single_image_inference(
            model=model,
            device=str(device),
            **common_kwargs,
        )

    display(pd.DataFrame([inference_output.to_summary_dict()]))
    display(pd.DataFrame(list(inference_output.point_lux.items()), columns=['point_name', 'predicted_lux']))
    print(json.dumps(inference_output.to_summary_dict(), indent=2))

    if inference_output.artifacts.prediction_overview_image:
        display(NotebookImage(filename=inference_output.artifacts.prediction_overview_image))


## 22. Next Steps

This notebook is the clean accessible-datasets workflow:
- NYU from Kaggle
- no MIT Intrinsic Images
- training, validation, testing, checkpointing, ONNX export, and single-image inference in one file

To improve real hallway performance, focus next on better custom hallway supervision: more lux maps, point targets, floor masks, and measured power data.
